In [ ]:
import logging
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib.font_manager').disabled = True
logging.getLogger('PIL.PngImagePlugin').disabled = True
logging.getLogger('matplotlib.mathtext').disabled = True
logger = logging.getLogger(__name__)

## Use basis states as input states

In [1]:
from solve_mix import *
from flow.plots import *
from flow.plots import plot_total_prob, plot_success_prob

In [ ]:
# TODO
# Simple tests or solver comparison?
# noise_level = 0.3
# case_id = f"{si.case_id}_mix_{noise_level}"


In [ ]:
problem = ProblemSpec(num_qubits=2, num_states=3, seed=42, state_type="densitymatrix")

In [ ]:
tracemalloc.start()

# Generate basis states
states = [1,2,3]
states[0] = np.array([1, 0, 0, 0], dtype=np.complex128)
states[1] = np.array([0, 1, 0, 0], dtype=np.complex128)
states[2] = np.array([0, 0, 1, 0], dtype=np.complex128)

## dense_states, disturbance_states, combined_states = ProblemSpec.gen_noisy_states(
##     num_qubits=2,
##     num_states=3,
##     seeds=get_random_seeds(3, seed=42),
##     noise_level=0.1,
##     noise_rank=2,
## )

# Convert states to DensityMatrix
dense_states = [DensityMatrix(states[_]) for _ in range(problem.num_states)]

disturbance_states = [
    DensityMatrix(ProblemSpec.depolarizing_noise_channel(2))
    for _ in range(3)
]

##
## print("dense_states")
## print(dense_states)
## print("disturbance_states")
## print(disturbance_states)
## print("combined_states")
## print(combined_states)
# Ideal, but use the new formulation
problem.set_states(state_type="densitymatrix", states=dense_states, overwrite=True)
print("dense_states")
print(problem.states[0].data)

# Noisy
problem.set_states(state_type="densitymatrix", states=combined_states, overwrite=True)

print("combined_states")
print(problem.states[0].data)

# TODO Different density matrices with noise
# noise_levels = [0.01 * i for i in range(1, 11)]
noise_levels = [0.01 * i for i in range(1, 3)]
# noise_levels = [0.1]
# Collect results
x_axis = []
y_axis = []
z_axis = []
# Parameters to draw the plots
# alpha, beta, prior_prob, total_success_prob, noise_level
# params = [0.1 ** (3 - 0.25 * i) for i in range(10)]
params = [0.5 - 0.04 * i for i in range(10)]
# params = [0.5 - 0.02 * i for i in range(20)]
x_axis = params
y_axis = params
for i in noise_levels:
    t_points = []
    s_points = []
    noise_level = i
    combined_states = [
        (1 - noise_level) * dense_states[_].data
        + noise_level * disturbance_states[_].data
        for _ in range(problem.num_states)
    ]

    combined_states = [
        DensityMatrix(combined_states[_]) for _ in range(problem.num_states)
    ]
    problem.set_states(
        state_type="densitymatrix", states=combined_states, overwrite=True
    )

    for a in params:
        for b in params:
            # Postprocessing
            result = apply_crossQD(
                problem_spec=problem,
                alpha=[a] * 3,
                beta=[b] * 3,
                noise_level=noise_level,
            )
            try:
                povm, total, p_d, inc_prob = result
                ## z_axis.append(p_d)
                ## points.append((a, b, p_d))
                # z_axis.append(total)
                t_points.append((a, b, total))
                s_points.append((a, b, p_d))
            except:
                # z_axis.append(0)
                t_points.append((a, b, 0))
                s_points.append((a, b, 0))
                print("The solver did not return a tuple.")
            print()

    # TODO draw a plot of total success probability vs. threshold
    # TODO draw with prior probabilities
    fig = plt.figure(dpi=900)
    fig.set_figwidth(6)
    fig.set_figheight(4.8)
    fig.set_size_inches(8, 5)
    # z_axis = np.array(z_axis).reshape(5, 5)
    # z_axis = np.array(z_axis).reshape(10, 10)
    # save_3dplot(x_axis, y_axis, z_axis, fig)
    plot_total_prob(t_points, tag=f"noise_{noise_level}", noise_level=noise_level)
    plot_success_prob(s_points, tag=f"noise_{noise_level}", noise_level=noise_level)
    t_arr = []
    for t in t_points:
        t_arr.append(t[2])
    # Dump t_arr
    np.save(f"t_arr_{noise_level}.npy", t_arr)
    # DO the same for s_points
    s_arr = []
    for s in s_points:
        s_arr.append(s[2])
    np.save(f"s_arr_{noise_level}.npy", s_arr)
        
    # save_3dplot(points, fig)

logger.info(f"Memory (current, peak, in bytes) = {tracemalloc.get_traced_memory()}")
tracemalloc.stop()
# np.save(f"povm_{case_id}.npy", povm)
# logger.info(f"The POVM is saved to povm_{case_id}.npy")
# TODO Remember the remaining operators

INFO:problem_spec:Use the seeds from the keyword arguments
INFO:problem_spec:3 random 2-qubit noisy states are generated
INFO:solve_mix:The prior probabilities is set to uniform (n = 3)
INFO:solve_mix:The threshold probabilities (alpha) are set to [0.5, 0.5, 0.5]
INFO:solve_mix:The threshold probabilities (beta) are set to [0.5, 0.5, 0.5]


dense_states
[[ 4.5474e-01-3.1219e-18j  1.7136e-01+2.4159e-02j -2.9172e-01+2.0664e-01j
  -4.0758e-02+2.9757e-01j]
 [ 1.7136e-01-2.4159e-02j  6.5855e-02+1.7618e-19j -9.8948e-02+9.3363e-02j
   4.5043e-04+1.1430e-01j]
 [-2.9172e-01-2.0664e-01j -9.8948e-02-9.3363e-02j  2.8103e-01-1.3130e-17j
   1.6136e-01-1.7237e-01j]
 [-4.0758e-02-2.9757e-01j  4.5043e-04-1.1430e-01j  1.6136e-01+1.7237e-01j
   1.9837e-01-7.7384e-19j]]
combined_states
[[ 4.3427e-01-2.8097e-18j  1.5422e-01+2.1743e-02j -2.6255e-01+1.8597e-01j
  -3.6682e-02+2.6781e-01j]
 [ 1.5422e-01-2.1743e-02j  8.4269e-02+1.5856e-19j -8.9053e-02+8.4027e-02j
   4.0539e-04+1.0287e-01j]
 [-2.6255e-01-1.8597e-01j -8.9053e-02-8.4027e-02j  2.7793e-01-1.1817e-17j
   1.4523e-01-1.5513e-01j]
 [-3.6682e-02-2.6781e-01j  4.0539e-04-1.0287e-01j  1.4523e-01+1.5513e-01j
   2.0353e-01-6.9646e-19j]]


INFO:solve_mix:CVXPY returns optimal


Result = 0.8679473543643352
CVXPY returns optimal
Solution for PI_0 =
[[ 0.5665+0.j      0.129 +0.0696j -0.1492+0.3342j  0.0838+0.214j ]
 [ 0.129 -0.0696j  0.1563+0.j     -0.021 +0.1754j -0.1078+0.0888j]
 [-0.1492-0.3342j -0.021 -0.1754j  0.2985+0.j      0.1749-0.0129j]
 [ 0.0838-0.214j  -0.1078-0.0888j  0.1749+0.0129j  0.3128+0.j    ]]
Solution for PI_1 =
[[ 0.1763+0.j      0.0558+0.1589j  0.1372-0.2442j  0.0334-0.0009j]
 [ 0.0558-0.1589j  0.295 +0.j     -0.0542-0.3133j -0.1089+0.0847j]
 [ 0.1372+0.2442j -0.0542+0.3133j  0.651 +0.j     -0.1776+0.0514j]
 [ 0.0334+0.0009j -0.1089-0.0847j -0.1776-0.0514j  0.2101+0.j    ]]
Solution for PI_2 =
[[ 0.2572+0.j     -0.1848-0.2285j  0.0121-0.0899j -0.1173-0.2131j]
 [-0.1848+0.2285j  0.5487+0.j      0.0752+0.1378j  0.2167-0.1734j]
 [ 0.0121+0.0899j  0.0752-0.1378j  0.0504+0.j      0.0027-0.0385j]
 [-0.1173+0.2131j  0.2167+0.1734j  0.0027+0.0385j  0.4771+0.j    ]]
Solution for PI_3 =
[[ 4.0846e-08+0.0000e+00j  5.3359e-08+5.4763e-08j -2.4885e-08+5

DEBUG:matplotlib.colorbar:locator: <matplotlib.ticker.AutoLocator object at 0x7f8d5785ce30>
INFO:flow.plot_results:The heatmap is saved as seaborn_heatmap_3_crossQD_$\alpha$0.500000_$\beta$0.500000_l0.010000.png
INFO:solve_mix:The prior probabilities is set to uniform (n = 3)
INFO:solve_mix:The threshold probabilities (alpha) are set to [0.5, 0.5, 0.5]
INFO:solve_mix:The threshold probabilities (beta) are set to [0.46, 0.46, 0.46]


...

DEBUG:matplotlib.colorbar:locator: <matplotlib.ticker.AutoLocator object at 0x7f8d56b8f470>
INFO:flow.plot_results:The heatmap is saved as seaborn_heatmap_3_crossQD_$\alpha$0.140000_$\beta$0.180000_l0.020000.png
INFO:solve_mix:The prior probabilities is set to uniform (n = 3)
INFO:solve_mix:The threshold probabilities (alpha) are set to [0.14, 0.14, 0.14]
INFO:solve_mix:The threshold probabilities (beta) are set to [0.14, 0.14, 0.14]


INFO:solve_mix:CVXPY returns optimal


Result = 0.738345332955765
CVXPY returns optimal
Solution for PI_0 =
[[ 0.4935+0.j      0.185 +0.0025j -0.0747+0.3493j  0.0177+0.2336j]
 [ 0.185 -0.0025j  0.0693+0.j     -0.0263+0.1313j  0.0078+0.0875j]
 [-0.0747-0.3493j -0.0263-0.1313j  0.2586+0.j      0.1627-0.0479j]
 [ 0.0177-0.2336j  0.0078-0.0875j  0.1627+0.0479j  0.1112+0.j    ]]
Solution for PI_1 =
[[ 0.0992+0.j      0.0945+0.1119j  0.1613-0.1757j -0.0295+0.0518j]
 [ 0.0945-0.1119j  0.2163+0.j     -0.0446-0.3495j  0.0303+0.0827j]
 [ 0.1613+0.1757j -0.0446+0.3495j  0.5738+0.j     -0.1398+0.032j ]
 [-0.0295-0.0518j  0.0303-0.0827j -0.1398-0.032j   0.0358+0.j    ]]
Solution for PI_2 =
[[ 0.1698+0.j     -0.0915-0.2562j  0.0284-0.0368j -0.1429-0.1453j]
 [-0.0915+0.2562j  0.4359+0.j      0.0402+0.0627j  0.2963-0.1374j]
 [ 0.0284+0.0368j  0.0402-0.0627j  0.0127+0.j      0.0076-0.0553j]
 [-0.1429+0.1453j  0.2963+0.1374j  0.0076+0.0553j  0.2447+0.j    ]]
Solution for PI_3 =
[[ 0.2375+0.j     -0.188 +0.1419j -0.115 -0.1368j  0.1547-0.1401

DEBUG:matplotlib.colorbar:locator: <matplotlib.ticker.AutoLocator object at 0x7f8d4db7f530>
INFO:flow.plot_results:The heatmap is saved as seaborn_heatmap_3_crossQD_$\alpha$0.140000_$\beta$0.140000_l0.020000.png


INFO:__main__:Memory (current, peak, in bytes) = (25223677, 28485185)


In [3]:
plt.show()

<function matplotlib.pyplot.show(*, block=None)>

In [ ]:
dense_states, disturbance_states, combined_states = ProblemSpec.gen_noisy_states(
    num_qubits=si.nq,
    num_states=si.ns,
    seeds=get_random_seeds(si.ns, seed=si.state_seed),
    noise_level=noise_level,
    noise_rank=2,
)
